# Combine Two Microarray Parquet Files

This notebook loads two standardized microarray parquet files, unions their SNP/variant sets by `# rsid`, and reports genotype mismatches on intersecting variants.

It assumes the parquet files contain at least:
- `# rsid` or a similar rsid column
- a genotype column such as `genotype`, `GT`, or `call`
- optional metadata columns like `chromosome` and `position`

In [5]:
import pandas as pd
from pathlib import Path


In [7]:
file1_path = Path("/home/frederik/github_projects/SNPster/data pipeline/harmonizing_module/test_data/IMPID29.chr22.standardizedMicroarray.parquet")
file2_path = Path("/home/frederik/github_projects/SNPster/data pipeline/harmonizing_module/test_data/IMPID62.chr22.standardizedMicroarray.parquet")

def load_microarray_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet file: {path}")
    return pd.read_parquet(path, engine='pyarrow')  # Specify engine to avoid ambiguity




In [ ]:
data1 = load_microarray_parquet(file1_path)
data2 = load_microarray_parquet(file2_path)
# columns: # rsid, chromosome, position, genotype

# Concatenate the two dataframes
full_df = pd.concat([data1, data2], ignore_index=True)

# Check duplicate rsids for conflicting genotypes
duplicate_rsids = full_df[full_df.duplicated(subset="# rsid", keep=False)]

conflicting_rsids = (
    duplicate_rsids
    .groupby("# rsid")["genotype"]
    .nunique()
)

conflicting_rsids = conflicting_rsids[conflicting_rsids > 1]

if not conflicting_rsids.empty:
    conflicts = full_df[
        full_df["# rsid"].isin(conflicting_rsids.index)
    ].sort_values("# rsid")

    raise ValueError(
        f"Conflicting genotypes found for {len(conflicting_rsids)} rsids:\n"
        f"{conflicts.to_string(index=False)} \n"
        f"{len(conflicting_rsids)} rsids have conflicting genotypes. Please resolve the conflicts before proceeding."
    )

# Duplicates are identical, so keep only one copy of each rsid
full_df = full_df.drop_duplicates(subset="# rsid", keep="first").reset_index(drop=True)

ValueError: Conflicting genotypes found for 1217 rsids:
    # rsid chromosome  position genotype
 rs1001445         22  24772430       CC
 rs1001445         22  24772430       AC
 rs1003480         22  30950766       AG
 rs1003480         22  30950766       AA
 rs1003694         22  37143088       CT
 rs1003694         22  37143088       CC
 rs1004529         22  44056438       AA
 rs1004529         22  44056438       GG
 rs1005640         22  20434787       CC
 rs1005640         22  20434787       CT
 rs1007888         22  23898914       CC
 rs1007888         22  23898914       TT
 rs1009385         22  33853446       AG
 rs1009385         22  33853446       GG
 rs1009730         22  33864750       CC
 rs1009730         22  33864750       CT
 rs1012068         22  31869917       TT
 rs1012068         22  31869917       GT
 rs1014971         22  38936618       TT
 rs1014971         22  38936618       CT
 rs1015067         22  26753090       TT
 rs1015067         22  26753090       CT
 

In [ ]:
import pandas as pd
from pathlib import Path

# Update these paths before running.
file1_path = Path("/home/frederik/github_projects/SNPster/data pipeline/harmonizing_module/test_data/IMPID29.chr22.standardizedMicroarray.parquet")
file2_path = Path("/path/to/second_microarray.parquet")
output_path = Path("/home/frederik/github_projects/SNPster/data pipeline/util_dev/ultimate_microarray.parquet")


def load_microarray_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet file: {path}")
    return pd.read_parquet(path)


def find_identifier_column(df: pd.DataFrame) -> str:
    candidates = ["# rsid", "rsid", "SNP", "snp", "ID", "id"]
    for column in candidates:
        if column in df.columns:
            return column
    raise ValueError(f"Could not find an rsid/SNP identifier column. Available columns: {list(df.columns)}")


def find_genotype_column(df: pd.DataFrame) -> str:
    candidates = ["genotype", "GT", "call", "genotype_call", "allele", "genotype_value"]
    for column in candidates:
        if column in df.columns:
            return column
    raise ValueError(f"Could not find a genotype column. Available columns: {list(df.columns)}")


def normalize_microarray(df: pd.DataFrame, source_label: str) -> pd.DataFrame:
    id_column = find_identifier_column(df)
    genotype_column = find_genotype_column(df)

    normalized = df.copy()
    normalized = normalized.rename(columns={id_column: "variant_id", genotype_column: f"genotype_{source_label}"})
    normalized = normalized.drop_duplicates(subset=["variant_id"], keep="first")
    return normalized


In [ ]:
# Uncomment if your notebook kernel does not have parquet support yet:
# %pip install pyarrow

left = normalize_microarray(load_microarray_parquet(file1_path), "file1")
right = normalize_microarray(load_microarray_parquet(file2_path), "file2")

combined = left.merge(
    right,
    on="variant_id",
    how="outer",
    suffixes=("_file1", "_file2"),
    indicator=True,
)

left_genotype_col = "genotype_file1"
right_genotype_col = "genotype_file2"

combined["genotype_mismatch"] = (
    combined[left_genotype_col].notna()
    & combined[right_genotype_col].notna()
    & (combined[left_genotype_col] != combined[right_genotype_col])
)

discrepancies = combined.loc[
    combined["genotype_mismatch"],
    ["variant_id", left_genotype_col, right_genotype_col, "_merge"]
].copy()

ultimate_microarray = pd.DataFrame({"variant_id": combined["variant_id"]})

all_base_columns = set()
all_base_columns.update(col[:-6] for col in combined.columns if col.endswith("_file1"))
all_base_columns.update(col[:-6] for col in combined.columns if col.endswith("_file2"))
all_base_columns.discard("variant_id")

for column in sorted(all_base_columns):
    file1_column = f"{column}_file1"
    file2_column = f"{column}_file2"

    if file1_column in combined.columns and file2_column in combined.columns:
        ultimate_microarray[column] = combined[file1_column].combine_first(combined[file2_column])
    elif file1_column in combined.columns:
        ultimate_microarray[column] = combined[file1_column]
    elif file2_column in combined.columns:
        ultimate_microarray[column] = combined[file2_column]

ultimate_microarray["genotype"] = combined[left_genotype_col].combine_first(combined[right_genotype_col])
ultimate_microarray["genotype_mismatch"] = combined["genotype_mismatch"]
ultimate_microarray["source"] = combined["_merge"]

sort_columns = [column for column in ["chromosome", "position", "variant_id"] if column in ultimate_microarray.columns]
if sort_columns:
    ultimate_microarray = ultimate_microarray.sort_values(sort_columns, kind="mergesort").reset_index(drop=True)
else:
    ultimate_microarray = ultimate_microarray.sort_values("variant_id", kind="mergesort").reset_index(drop=True)

print(f"File 1 variants: {len(left)}")
print(f"File 2 variants: {len(right)}")
print(f"Union variants: {len(ultimate_microarray)}")
print(f"Intersecting variants: {(combined['_merge'] == 'both').sum()}")
print(f"Genotype mismatches: {len(discrepancies)}")

ultimate_microarray.to_parquet(output_path, index=False)
print(f"Wrote combined parquet to: {output_path}")

discrepancies